In [23]:
# Setup

import json
import time
import os
import csv
import re
from typing import List, Literal
from google import genai
from pydantic import BaseModel
from dotenv import load_dotenv

class Sentence(BaseModel):
    marked_sentence: str        # e.g., "He <accused> him *of* theft."
    translation: str            # Traditional Chinese translation

class PosEntry(BaseModel):
    word_or_collocation: str    # e.g. accusation, accuse of
    explanation: str            # Usage/Grammar note in Traditional Chinese
    sentences: List[Sentence]

class WordResult(BaseModel):
    headword: str               # e.g. accuse
    explanation: str            # Usage/Grammar note in Traditional Chinese
    entries: List[PosEntry]     # One word can have multiple POS entries
    related_forms: List[str]    # e.g., "accused"
    
class BatchWordResult(BaseModel):
    results: List[WordResult]
    
load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")

if not API_KEY:
    raise ValueError("Please set the GEMINI_API_KEY environment variable.")

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [ ]:
# Testing

response = client.models.generate_content(
    model="gemini-2.5-flash", contents="Explain how AI works in a few words"
)
print(response.text)

AI learns from data to identify patterns and make intelligent decisions.


In [9]:
example_response = {
  "headword": "account",
  "explanation": "這個字常考「帳戶」、「解釋」或「考慮」的用法，尤其是介係詞的搭配。",
  "entries": [
    {
      "word_or_collocation": "account",
      "explanation": "指「帳戶」或「敘述」。在閱讀測驗中常出現 eyewitness account (目擊者描述)。",
      "sentences": [
        {
          "marked_sentence": "The survivor provided a detailed <account> *of* the accident, helping the police understand what had happened on the highway.",
          "translation": "倖存者提供了關於事故的詳細描述，幫助警方了解高速公路上發生了什麼事。"
        }
      ]
    },
    {
      "word_or_collocation": "take...into account",
      "explanation": "「常用於 take ... into account，意為「將...考慮進去」。介係詞固定用 into。",
      "sentences": [
        {
          "marked_sentence": "When planning the graduation trip, the committee *took* the students' safety *into* <account> to avoid any potential accidents.",
          "translation": "在規劃畢業旅行時，委員會將學生的安全考慮進去，以避免任何潛在的事故。"
        }
      ]
    },
    {
      "word_or_collocation": "account for",
      "explanation": "動詞片語，意為「解釋」或「佔...（比例）」。",
      "sentences": [
        {
          "marked_sentence": "Heavy rain and thick fog <account> *for* the delay of more than twenty international flights at the airport this morning.",
          "translation": "大雨和濃霧解釋了今天早上機場二十多個國際航班延誤的原因。"
        }
      ]
    }
  ],
  "related_forms": ["accounts (pl.)"]
}


result = WordResult.model_validate(example_response)

result

WordResult(headword='account', explanation='這個字常考「帳戶」、「解釋」或「考慮」的用法，尤其是介係詞的搭配。', entries=[PosEntry(word_or_collocation='account', explanation='指「帳戶」或「敘述」。在閱讀測驗中常出現 eyewitness account (目擊者描述)。', sentences=[Sentence(marked_sentence='The survivor provided a detailed <account> *of* the accident, helping the police understand what had happened on the highway.', translation='倖存者提供了關於事故的詳細描述，幫助警方了解高速公路上發生了什麼事。')]), PosEntry(word_or_collocation='take...into account', explanation='「常用於 take ... into account，意為「將...考慮進去」。介係詞固定用 into。', sentences=[Sentence(marked_sentence="When planning the graduation trip, the committee *took* the students' safety *into* <account> to avoid any potential accidents.", translation='在規劃畢業旅行時，委員會將學生的安全考慮進去，以避免任何潛在的事故。')]), PosEntry(word_or_collocation='account for', explanation='動詞片語，意為「解釋」或「佔...（比例）」。', sentences=[Sentence(marked_sentence='Heavy rain and thick fog <account> *for* the delay of more than twenty international flights at the airport this morning.', trans

In [16]:
def generate_data(batch_words: list[str]) -> BatchWordResult:
    prompt = f"""
    Act as a Taiwan GSAT English teacher.
    
    For each word, provide entries for ALL its common Parts of Speech (POS).
    
    Rules:

    - headword: The base form of the word.
    - word_or_collocation: The word related to the headword or a collocation involving the headword and other members.
    - explanation: High-value GSAT usage note (Traditional Chinese).
    - marked_sentence:
       - Length: 15-25 words. 
       - Context: Use academic, social, or school-life themes common in GSAT.
       - Marking: Use <> for the headword (only a single word, the word that's related to the headword) and ** for the key prepositions or collocations worth testing in a cloze test (if any, possibly more than one).
    - translation: Traditional Chinese.
    - related_forms: List[str] of relevant word family members, like verb conjugations or the noun form.
    - i+1 principle: Use clear context so the target word's meaning is obvious.
    
    Here's an example: {example_response}
    
    Here are the words: {batch_words}
    """
    
    # Using the latest 2026 SDK 'generate' method
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config={
            "response_mime_type": "application/json",
            "response_schema": BatchWordResult,
            "temperature": 0.2, # Low temperature for more deterministic output
        }
    )
    
    return response.parsed

In [11]:
# Loading word list

origin = "data/vocabulary"

# start from level 3
level = 3
word_list = []

with open(f"{origin}/level{level}.txt", "r", encoding="utf-8") as f:
    word_list = [line.strip() for line in f.readlines()]

print(f"Loaded {len(word_list)} words from level {level}.")
print(word_list[:5])

Loaded 1002 words from level 3.
['aboard prep./adv.', 'acceptable adj.', 'accurate adj.', 'ache n./v.', 'achieve(ment) v./(n.)']


In [ ]:
# Test run with first 5 words

words = word_list[:5]
batch_results = generate_data(words)
print(batch_results)

results=[WordResult(headword='aboard', explanation='這個字最常見的用法是作為介系詞，指「在...船上/飛機上/火車上」，或作為副詞，指「上船/飛機/火車」。', entries=[PosEntry(word_or_collocation='aboard', explanation='作為介系詞，表示「在（船、飛機、火車、公車等）上」。', sentences=[Sentence(marked_sentence='All passengers must show their boarding passes before getting <aboard> the airplane for the international flight.', translation='所有旅客在登上飛機搭乘國際班機前，都必須出示登機證。')]), PosEntry(word_or_collocation='come aboard', explanation='「上船（或飛機、火車等）」。', sentences=[Sentence(marked_sentence='The captain invited the distinguished guests to <come> <aboard> the cruise ship to experience the luxury amenities.', translation='船長邀請貴賓們上郵輪體驗豪華設施。')]), PosEntry(word_or_collocation='go aboard', explanation='「上船（或飛機、火車等）」。', sentences=[Sentence(marked_sentence='After the safety briefing, all the crew members were instructed to <go> <aboard> the research vessel for the expedition.', translation='安全簡報後，所有船員被指示上研究船準備探險。')])], related_forms=[]), WordResult(headword='acceptable', explanation='

In [ ]:
words = word_list

existing_words = set()
if os.path.exists("raw_gsat_data.tsv"):
    with open("raw_gsat_data.tsv", "r", encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        next(reader, None)  # Skip header
        for row in reader:
            if row: existing_words.add(row[0])



# filter out words already processed
words_to_process = []

for w in words:
    match = re.match(r"^([^(\s]+)", w)
    if match:
        headword = match.group(1)
        if headword not in existing_words:
            words_to_process.append(w)

print(existing_words)
print(words_to_process[:5])
print(f"Resuming: {len(existing_words)} already done. {len(words_to_process)} remaining.")

{'award', 'ash', 'approve', 'aside', 'agriculture', 'announce', 'amaze', 'awaken', 'advantage', 'apologize', 'arrest', 'aboard', 'angel', 'appeal', 'alley', 'alphabet', 'attract', 'ambassador', 'attitude', 'audience', 'awkward', 'automobile', 'additional', 'anyhow', 'ambition', 'adventure', 'assume', 'background', 'anxious', 'advise', 'ache', 'afford', 'avenue', 'almond', 'airline', 'advanced', 'adviser', 'apron', 'automatic', 'aware', 'assist', 'attractive', 'advertise', 'awake', 'assistant', 'armed', 'accurate', 'athlete', 'afterward', 'awful', 'acceptable', 'achieve', 'apart', 'ambulance', 'admire'}
['adviser/advisor n.', 'afterward/afterwards adv.', 'automobile/auto n.', 'bacon n.', 'bacteria n.']
Resuming: 55 already done. 950 remaining.


In [ ]:
# Processing loop



# open in APPEND mode ("a")
with open("raw_gsat_data.tsv", "a", encoding="utf-8", newline="") as f:
    writer = csv.writer(f, delimiter="\t")
    
    # Write header only if file is new
    if os.path.getsize("raw_gsat_data.tsv") == 0:
        writer.writerow(["headword", "response"])

    chunk_size = 5
    for i in range(0, len(words_to_process), chunk_size):
        chunk = words_to_process[i:i+chunk_size]
        print(f"Processing chunk {i//chunk_size + 1} out of {len(words_to_process)//chunk_size + 1}...")
        
        try:
            batch_results = generate_data(chunk)
            results = batch_results.results
            
            for r in results:
                # Write to disk IMMEDIATELY
                writer.writerow([
                    r.headword, 
                    r.model_dump_json()
                ])
            
            # Flush the buffer to ensure it's written to the physical disk
            f.flush()
            
            # Rate limiting
            time.sleep(5) 
            
        except Exception as e:
            print(f"Error at {chunk}: {e}")
            print("Current progress is saved. Waiting 30s before retry...")
            time.sleep(30)

Resuming: 55 already done. 1002 remaining.
Processing chunk 1 out of 201...
Processing chunk 2 out of 201...
Error at ['additional adj.', 'admire v.', 'advanced adj.', 'advantage n.', 'adventure n.']: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 39.337809911s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelang

KeyboardInterrupt: 